# CSA epoch 9 SCST + XE pilot, 500 updates

Attach visual-cache, mscoco-2014, csa-prompt-full-karpathy-v175 and resume Output containing best.bin epoch 9. T4 x2, Internet, Save & Run All. Select 4000 TRAIN images with seed 42. Adam 5e-6, global batch 8, XE plus .05 SCST. Greedy baseline and sampled caption use the current policy. Evaluate top1 beam5 on VALID5000. Keep pilot_checkpoint.bin and send csa_per_caption_scst_xe_pilot500_valid5000_review.zip.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='aae9afc7966ee4aa36dd67c4b4839ff788e775fe'
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
PROMPT_TEST_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
CHECKPOINT_ROOT='' # Optional mounted best.bin or folder. Blank scans attached inputs.
WALL_HOURS=3 # STOP cap including CPU metrics, not ETA or billing.
OUTPUT=Path('/kaggle/working/csa_per_caption_scst_xe_pilot500_valid5000')
for p in (VISUAL_CACHE,COCO_JSON,PROMPT_TEST_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct original inputs: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve existing captions/output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for TEST patches/checkpoint/CPU scorer assets')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch199-test-',dir='/tmp'));DEPS=SOURCE/'deps';CLI=SOURCE/'cli';CHAIR=SOURCE/'chair';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(semantic=True,prompt_test_root=str(PROMPT_TEST_ROOT),output=str(OUTPUT),stage=str(SOURCE/'test_patches'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),checkpoint_root=CHECKPOINT_ROOT,chair=str(CHAIR),worker_seconds=1800)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline before download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned input: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

TRAIN_REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
TRAIN_PINS={'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09'}

os.environ.update(TOKENIZERS_PARALLELISM='false',USE_TF='0',USE_FLAX='0',HF_HUB_DISABLE_XET='1')

cfg.update(train_stage=str(SOURCE/'pilot_train'),train_seconds=3600)
write_config()


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('TEST parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('TEST/CPU metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('TEST parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
ROUTING_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output('csa_structure_per_caption_full_xe10_resume124177',path=directory)\n    request=endpoint.call_args.args[0]\n    assert (request.user_name,request.kernel_slug,request.version_label)==('ducanh2403','kltnnnnnff','v9')\nprint('Exact own9 API routing mock PASS; no network.',flush=True)\n"
complete=False
try:
    for path,name in MAPPING.items():download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+path,HASHES[path],SOURCE/name)
    for path,digest in TRAIN_PINS.items():
        name='bank_layers.py' if path=='experiments/csa_paper/model.py' else Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+TRAIN_REVISION+'/'+path,digest,SOURCE/name)
    (SOURCE/'csa_per_caption_test_data.py').write_text('"""Fixed best9 TEST prompt/patch contracts; NumPy only, no model execution."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read,save,sha\nimport test199_data as shared\nfrom semantic_data import ARRAYS,record_arrays,checked_root\n\nHERE=Path(__file__).resolve().parent\nCHECKPOINT_SHA=\'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef\'\nVERSION=\'csa_structure_per_caption_full_xe10_resume124177\'\n\n\ndef validate_lock(value):\n    baseline=valid_cohort()\n    if any(value[k]!=baseline[k] for k in (\'expected_test_ids\',\'test_id_sha256\',\'beam_width\',\'max_new_tokens\',\'length_normalization\',\'reranking\')):raise ValueError(\'Changed fixed TEST cohort/decode\')\n    if value[\'source_version\']!=VERSION or value[\'checkpoint_sha256\']!=CHECKPOINT_SHA or value[\'epoch\']!=9 or value[\'updates\']!=159399 or value[\'method\']!=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\':raise ValueError(\'Need fixed bank+prompt best9, not endpoint10\')\n    return value\n\n\ndef valid_cohort():\n    import hashlib\n    value=shared.lock()\n    value.update(expected_test_ids=VALID_IDS,test_id_sha256=hashlib.sha256(json.dumps(VALID_IDS,separators=(\',\',\':\')).encode()).hexdigest())\n    return value\n\ndef lock():\n    value=valid_cohort();value.update(schema=\'csa_per_caption_best9_test_v1\',source_version=VERSION,checkpoint_sha256=CHECKPOINT_SHA,\n        training_commit=\'5eb012279e6fd0b5e47a56710ed5e3efa7449d2f\',epoch=9,updates=159399,method=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\')\n    return validate_lock(value)\n\n\ndef checkpoint(root):\n    root=Path(root);files=[root] if root.is_file() else sorted(root.rglob(\'best.bin\'))\n    matches=[p for p in files if p.is_file() and sha(p)==CHECKPOINT_SHA]\n    if not matches:raise ValueError(\'Missing exact version9 best.bin SHAb0c5bde0; no final/latest fallback\')\n    return matches[0]\n\n\ndef runtime_policy(training,current):\n    if training!=\'2.11.0+cu128\' or current not in (\'2.11.0+cu128\',\'2.11.0+cu130\'):raise RuntimeError(\'Unreviewed Torch build; keep log, do not blindly reinstall.\')\n    return dict(training_torch=training,inference_torch=current,cuda_wheel_changed=training!=current,\n        policy=\'Same publicTorch2.11.0 observed CUDA wheels; strictstate and actual GPU beam-path check required.\',bitwise_cross_build_equivalence_established=False)\n\n\ndef validate_checkpoint_state(ck):\n    s,i=ck[\'state\'],ck[\'identity\'];expected=lock()\n    if s[\'epoch\']!=10 or s[\'batch_cursor\']!=0 or s[\'phase\']!=\'train\' or s[\'updates\']!=159399 or s[\'caption_events\']!=5100723 or s[\'best_epoch\']!=9 or len(s[\'history\'])!=9:raise ValueError(\'Wrong best9 snapshot cursor\')\n    if i[\'source\'][\'commit\']!=expected[\'training_commit\'] or i[\'method\']!=expected[\'method\'] or i[\'epochs\']!=10 or i[\'global_batch\']!=32:raise ValueError(\'Wrong checkpoint architecture/training identity\')\n    if not {\'core.nouns\',\'core.verbs\'}<=set(ck[\'model\']):raise ValueError(\'Missing frozen TRAIN banks in checkpoint buffers\')\n    return expected\n\n\ndef configure_valid(cfg):\n    global VALID_IDS\n    if sha(cfg[\'dataset_json\'])!=shared.DATA_SHA:raise ValueError(\'Changed Karpathy metadata\')\n    def val_rows(ds, locked):\n        rows=sorted((r for r in ds[\'images\'] if r[\'split\']==\'val\'),key=lambda r:int(r[\'cocoid\']))\n        if [int(r[\'cocoid\']) for r in rows]!=locked[\'expected_test_ids\']:raise ValueError(\'Wrong VALID IDs\')\n        return rows\n    VALID_IDS=sorted(int(r[\'cocoid\']) for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\']==\'val\')\n    if len(VALID_IDS)!=5000 or len(set(VALID_IDS))!=5000:raise ValueError(\'Need VALID5000\')\n    shared.selected_test_rows=val_rows\n\ndef prompt_inputs(cfg):\n    configure_valid(cfg)\n    inv=read(HERE/\'FULL_PROMPT_FRESH_V175_RESULT.json\')[\'transport_inventory\'][\'cache_files\']\n    wanted={n:inv[n] for n in (\'cache_receipt.json\',\'cohort.json\',\'prompt_links.jsonl\',\'prompt_texts.json\',\'prompt_embeddings.npy\')}\n    root=checked_root(cfg[\'prompt_test_root\'],\'cache_receipt.json\',wanted)\n    cohort=read(root/\'cohort.json\')\n    if cohort[\'test\'] or sorted(cohort[\'val\'])!=VALID_IDS:raise ValueError(\'Wrong prompt175 VALID cohort\')\n    texts=read(root/\'prompt_texts.json\')\n    vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False)\n    if texts!=sorted(set(texts)) or vectors.shape!=(410,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all():raise ValueError(\'Changed prompt175 embeddings\')\n    with (root/\'prompt_links.jsonl\').open(encoding=\'utf8\') as stream:\n        rows=[r for r in map(json.loads,stream) if r[\'group\']==\'val\']\n    if sorted(r[\'coco_id\'] for r in rows)!=VALID_IDS:raise ValueError(\'Wrong VALID prompt links\')\n    return rows,texts,vectors,wanted\n\ndef prepare(cfg):\n    rows,texts,vectors,seals=prompt_inputs(cfg)\n    for row in rows:record_arrays(row,texts)\n    shared.prepare(cfg,locked=lock(),weight_path=checkpoint(cfg[\'checkpoint_root\']))\n    stage_prompts(cfg,rows,texts,vectors,seals)\n\n\ndef stage_prompts(cfg,rows,texts,vectors,seals):\n    encoded=[record_arrays(row,texts) for row in rows]\n    stage=Path(cfg[\'stage\']);out=Path(cfg[\'output\']);ids=np.load(stage/\'ids.npy\',allow_pickle=False);lookup={r[\'coco_id\']:j for j,r in enumerate(rows)}\n    order=[lookup[int(i)] for i in ids]\n    for key in ARRAYS:np.save(stage/(\'sem_\'+key+\'.npy\'),np.stack([encoded[j][key] for j in order]))\n    np.save(stage/\'sem_text_vectors.npy\',vectors)\n    receipt=read(out/\'stage_receipt.json\');receipt[\'method\']=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\';receipt[\'prompt175_files_sha256\']=seals\n    receipt[\'stage_files_sha256\'].update({p.name:sha(p) for p in stage.glob(\'sem_*.npy\')});save(out/\'stage_receipt.json\',receipt)\n    saved=read(out/\'test_lock.json\');saved[\'stage_receipt_sha256\']=sha(out/\'stage_receipt.json\');save(out/\'test_lock.json\',saved)\n    save(out/\'semantic_test_stage.json\',dict(images=5000,objects=sum(len(r[\'objects\']) for r in rows),relations=sum(len(r[\'relations\']) for r in rows),\n        text_vectors=410,geometry=\'same training record_arrays ROI box->fractional14x14patch support; endpoint IDs\',TRAIN_TEST_features_read=False,old_ROI_features_used=False,old_vocabulary_used=False))\n\n\ndef load(cfg,verify=True):\n    result=shared.load(cfg,verify=verify,validator=validate_lock)\n    if result[2].get(\'method\')!=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\':raise ValueError(\'Semantic TEST stage incomplete\')\n    return result\n',encoding='utf8')
    (SOURCE/'run_csa_per_caption_test.py').write_text('"""USER-RUN fixed199 inference and CPU scoring. No training mode."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom test199_data import prepare,load,lock,validate_lock,checkpoint,runtime_policy,CHECKPOINT_SHA,TEST_SIZE\nfrom saved_quality_data import check_rows,chair_counts\n\ndef contracts(cfg):\n    if cfg.get(\'semantic\'):\n        import csa_per_caption_test_data as module\n        module.configure_valid(cfg)\n    else:\n        import test199_data as module\n    return module\n\n\ndef checked_weights(cfg):\n    import torch\n    d=contracts(cfg)\n    if cfg.get(\'evaluate_pilot\'):\n        receipt=read(Path(cfg[\'output\'])/\'pilot_checkpoint_receipt.json\')\n        path=Path(cfg[\'output\'])/\'pilot_checkpoint.bin\'\n        if receipt[\'updates\']!=500 or receipt[\'source_sha256\']!=d.CHECKPOINT_SHA or sha(path)!=receipt[\'sha256\']:\n            raise ValueError(\'Wrong 500-update checkpoint\')\n        ck=torch.load(path,map_location=\'cpu\',weights_only=False)\n        if ck[\'updates\']!=500 or ck[\'source_sha256\']!=d.CHECKPOINT_SHA:\n            raise ValueError(\'Pilot state differs\')\n        return ck,d.runtime_policy(\'2.11.0+cu128\',str(torch.__version__))\n    d=contracts(cfg);ck=torch.load(d.checkpoint(cfg[\'checkpoint_root\']),map_location=\'cpu\',weights_only=False)\n    if cfg.get(\'semantic\'):\n        d.validate_checkpoint_state(ck)\n        return ck,d.runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n    if ck[\'state\'][\'updates\']!=177110 or ck[\'state\'][\'best_epoch\']!=10 or ck[\'identity\'][\'source\'][\'commit\']!=lock()[\'training_commit\']:raise ValueError(\'Unexpected fixed199 training state\')\n    return ck,runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n\ndef checkpoint_preflight(cfg):\n    ck,policy=checked_weights(cfg);del ck\n    save(Path(cfg[\'output\'])/\'checkpoint_runtime_preflight.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,runtime_policy=policy,models_executed=False))\n    print(\'Checkpoint/runtime metadata PASS:\',policy,flush=True)\n\ndef worker(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,check_paths,text_from_ids\n    from patch_data import batch as caption_batch\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Use T4x2\')\n    torch.set_num_threads(1);torch.cuda.set_device(rank);device=torch.device(\'cuda\',rank)\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    dist.init_process_group(\'nccl\',timeout=timedelta(seconds=1200));started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        d=contracts(cfg);data,words,receipt,locked=d.load(cfg,verify=rank==0);dist.barrier(device_ids=[rank])\n        ck,policy=checked_weights(cfg);save(out/f\'runtime_policy_rank{rank}.json\',policy)\n        context=None\n        if cfg.get(\'semantic\'):\n            from csa_patch_model import CSAPatchCaptioner\n            from semantic_data import SemanticContext\n            model=CSAPatchCaptioner(len(words),ck[\'model\'][\'core.nouns\'],ck[\'model\'][\'core.verbs\']);context=SemanticContext(cfg[\'stage\'])\n        else:model=PatchCaptioner(len(words))\n        model=model.to(device);model.load_state_dict(ck[\'model\'],strict=True);del ck;model.eval();rows=[];images=np.arange(TEST_SIZE)[rank::2]\n        with torch.inference_mode():\n            for offset in range(0,len(images),8):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST decode phase deadline\')\n                index=images[offset:offset+8];x=torch.from_numpy(np.asarray(data[\'patches\'][index],np.float32)).to(device)\n                memory=model.encode(x,context.images(index,device)) if context else model.encode(x);seq,scores=beam_search(model,memory,width=locked[\'beam_width\'],maximum=locked[\'max_new_tokens\'])\n                if offset==0:check_paths(model,memory,seq,scores)\n                for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                if offset%200==0:print(\'TEST decode rank\',rank,len(rows),\'/\',len(images),flush=True)\n            save(out/f\'captions_rank{rank}.json\',rows)\n            nll_sum=0.;tokens=0;ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n            for offset in range(0,len(ev),32):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST teacherNLL phase deadline; captions preserved\')\n                x,seq=caption_batch(data,ev[offset:offset+32]);x=torch.from_numpy(x).to(device);seq=torch.from_numpy(seq).to(device)\n                logits=model(x,seq[:,:-1],context.events(data,ev[offset:offset+32],device)) if context else model(x,seq[:,:-1]);nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n        if len(rows)!=2500:raise ValueError(\'Incomplete TEST rank\')\n        save(out/f\'worker{rank}.json\',dict(complete=True,rank=rank,images=len(rows),NLL_sum=nll_sum,NLL_tokens=tokens,seconds=time.monotonic()-started,training=False,checkpoint_sha256=read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'] if cfg.get(\'evaluate_pilot\') else d.CHECKPOINT_SHA,caption_sha256=sha(out/f\'captions_rank{rank}.json\'),runtime_policy=policy,first_GPU_beam_path_check_passed=True))\n        dist.barrier(device_ids=[rank])\n    finally:\n        if dist.is_initialized():dist.destroy_process_group()\n\ndef inputs(cfg):\n    d=contracts(cfg);out=Path(cfg[\'output\']);locked=d.lock();saved=read(out/\'test_lock.json\');d.validate_lock(saved);receipt=read(out/\'stage_receipt.json\')\n    if saved[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\') or sha(out/\'test_references.json\')!=receipt[\'output_files_sha256\'][\'test_references.json\']:raise ValueError(\'Changed TEST references/receipt\')\n    refs=read(out/\'test_references.json\');workers=[read(out/f\'worker{r}.json\') for r in (0,1)]\n    if not all(w[\'complete\'] and w[\'images\']==2500 and w[\'checkpoint_sha256\']==read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'] for w in workers):raise ValueError(\'Incomplete TEST model output\')\n    if workers[0][\'runtime_policy\']!=workers[1][\'runtime_policy\'] or not all(w[\'first_GPU_beam_path_check_passed\'] for w in workers):raise ValueError(\'Runtime/beam guards differ between ranks\')\n    rows=[]\n    for rank,w in enumerate(workers):\n        if sha(out/f\'captions_rank{rank}.json\')!=w[\'caption_sha256\']:raise ValueError(\'Changed TEST rank captions\')\n        rows+=read(out/f\'captions_rank{rank}.json\')\n    check_rows(rows,locked[\'expected_test_ids\'])\n    if len(refs)!=5000 or {r[\'coco_id\'] for r in refs}!=set(locked[\'expected_test_ids\']):raise ValueError(\'Wrong TEST references\')\n    return out,refs,rows,workers\n\ndef tokenize(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    out,refs,rows,workers=inputs(cfg);rows=sorted(rows,key=lambda r:r[\'coco_id\']);save(out/\'captions.json\',rows)\n    tok=PTBTokenizer();gts=tok.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=tok.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in rows})\n    save(out/\'reference_ptb.json\',gts);save(out/\'caption_ptb.json\',hyp)\n    save(out/\'scoring_lock.json\',dict(checkpoint_sha256=read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'],files_sha256={n:sha(out/n) for n in (\'captions.json\',\'test_references.json\',\'test_lock.json\',\'reference_ptb.json\',\'caption_ptb.json\')},training=False,reranking=False))\n\ndef score(cfg,mode):\n    out=Path(cfg[\'output\']);sealed=read(out/\'scoring_lock.json\')\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed scoring input \'+n)\n    gts=read(out/\'reference_ptb.json\');hyp=read(out/\'caption_ptb.json\')\n    if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError(\'Incomplete TEST scorer cohort\')\n    result={}\n    if mode==\'basic\':\n        from pycocoevalcap.bleu.bleu import Bleu\n        from pycocoevalcap.cider.cider import Cider\n        from pycocoevalcap.rouge.rouge import Rouge\n        b,_=Bleu(4).compute_score(gts,hyp);c,per=Cider().compute_score(gts,hyp);r,_=Rouge().compute_score(gts,hyp)\n        result=dict(BLEU_1=float(b[0]),BLEU_4=float(b[3]),CIDEr=float(c),ROUGE_L=float(r));save(out/\'cider_per_image.json\',[dict(coco_id=int(i),CIDEr=float(v)) for i,v in zip(gts,per)])\n    elif mode in (\'meteor\',\'spice\'):\n        if mode==\'meteor\':\n            from pycocoevalcap.meteor.meteor import Meteor\n            scorer=Meteor()\n        else:\n            from pycocoevalcap.spice.spice import Spice\n            scorer=Spice()\n        try:point,per=scorer.compute_score(gts,hyp)\n        finally:del scorer\n        if len(per)!=5000:raise ValueError(\'Incomplete \'+mode)\n        result={mode.upper():float(point)}\n        values=[float(x) for x in per] if mode==\'meteor\' else [float(x[\'All\'][\'f\']) for x in per]\n        if not all(math.isfinite(x) for x in values):raise ValueError(\'Nonfinite per-image \'+mode)\n        save(out/(mode+\'_per_image.json\'),[dict(coco_id=int(i),score=v) for i,v in zip(gts,values)])\n    elif mode==\'chair\':\n        rows=read(out/\'captions.json\');save(out/\'chair_input.json\',[dict(image_id=r[\'coco_id\'],caption=r[\'caption\']) for r in rows]);chair=Path(cfg[\'chair\'])\n        subprocess.run([sys.executable,\'-u\',chair/\'chair.py\',\'--cap_file\',out/\'chair_input.json\',\'--cache\',chair/\'chair.pkl\',\'--save_path\',out/\'chair_output.json\'],cwd=chair,check=True,timeout=600)\n        result,counts=chair_counts(read(out/\'chair_output.json\'),rows);save(out/\'chair_counts.json\',counts)\n    else:raise ValueError(mode)\n    if not all(math.isfinite(v) for v in result.values()):raise ValueError(\'Nonfinite metric\')\n    save(out/(mode+\'.json\'),dict(images=5000,metrics=result,input_sha256=sealed[\'files_sha256\']))\n\ndef combine(cfg):\n    out,refs,rows,workers=inputs(cfg);sealed=read(out/\'scoring_lock.json\');metrics={}\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed final scoring input \'+n)\n    for mode in (\'basic\',\'meteor\',\'spice\',\'chair\'):\n        p=read(out/(mode+\'.json\'))\n        if p[\'images\']!=5000 or p[\'input_sha256\']!=sealed[\'files_sha256\']:raise ValueError(\'Incomplete/changed metric phase\')\n        metrics.update(p[\'metrics\'])\n    required={\'BLEU_1\',\'BLEU_4\',\'METEOR\',\'ROUGE_L\',\'CIDEr\',\'SPICE\',\'CHAIRs\',\'CHAIRi\',\'Recall\'}\n    if set(metrics)!=required or not all(math.isfinite(v) for v in metrics.values()):raise ValueError(\'Need all eight metrics and recall\')\n    metrics[\'NLL\']=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers)\n    save(out/\'result.json\',dict(status=\'CSA_PER_CAPTION_BEST9_TEST5000_COMPLETE\' if cfg.get(\'semantic\') else \'CLIP_PATCH_199_TEST5000_COMPLETE\',images=5000,checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,source_version=contracts(cfg).lock()[\'source_version\'],epoch=contracts(cfg).lock()[\'epoch\'],metrics=metrics,display_x100={k:100*v for k,v in metrics.items() if k!=\'NLL\'},beam_width=5,max_new_tokens=20,training=False,reranking=False,test_tuning=False,prior_project_test_exposure=True,chair_counts=read(out/\'chair_counts.json\'),environment=read(out/\'environment.json\'),runtime_policy=workers[0][\'runtime_policy\']))\n    print(\'TEST5000 COMPLETE\',metrics,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'checkpoint_preflight\',\'prepare\',\'worker\',\'environment\',\'smoke\',\'tokenize\',\'basic\',\'meteor\',\'spice\',\'chair\',\'combine\'));p.add_argument(\'--config\',required=True);a=p.parse_args();cfg=read(a.config)\n    if a.mode in (\'environment\',\'smoke\',\'basic\',\'meteor\',\'spice\',\'chair\'):\n        import score_prompt_test_extra as cpu\n        if a.mode==\'environment\':cpu.environment(Path(cfg[\'output\']))\n        else:\n            cpu.java_path(Path(cfg[\'output\']))\n            if a.mode==\'smoke\':cpu.smoke(Path(cfg[\'output\']))\n            else:score(cfg,a.mode)\n    elif a.mode==\'tokenize\':\n        import score_prompt_test_extra as cpu\n        cpu.java_path(Path(cfg[\'output\']));tokenize(cfg)\n    elif a.mode==\'prepare\':contracts(cfg).prepare(cfg)\n    else:globals()[a.mode](cfg)\n',encoding='utf8')
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES,training_commit=TRAIN_REVISION,training_pins=TRAIN_PINS),indent=2))
    run([sys.executable,'-u',SOURCE/'test_test199.py'],120)
    run([sys.executable,'-u',SOURCE/'test_semantic_test.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)
    sys.path.insert(0,str(SOURCE))
    from csa_per_caption_test_data import checkpoint,VERSION,CHECKPOINT_SHA,prompt_inputs
    weights=checkpoint(CHECKPOINT_ROOT or '/kaggle/input')
    origin='mounted resume Output, verified by SHA256'
    cfg['checkpoint_root']=str(weights);write_config()
    download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/5eb012279e6fd0b5e47a56710ed5e3efa7449d2f/experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json','452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a',SOURCE/'FULL_PROMPT_FRESH_V175_RESULT.json')
    prompt_inputs(cfg)
    (OUTPUT/'checkpoint_source.json').write_text(json.dumps(dict(version=VERSION,sha256=CHECKPOINT_SHA,path=str(weights),origin=origin),indent=2))
    run([sys.executable,'-u',SOURCE/'run_csa_per_caption_test.py','checkpoint_preflight','--config',CONFIG],120)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2','nltk==3.9.2','future'],600)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],600)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],300)
    base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
    for name,digest in {'chair.py':'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d','chair.pkl':'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'}.items():download(base+name,digest,CHAIR/name)
    run([sys.executable,'-u',SOURCE/'run_csa_per_caption_test.py','environment','--config',CONFIG],1200)
    run([sys.executable,'-u',SOURCE/'run_csa_per_caption_test.py','smoke','--config',CONFIG],600)
    run([sys.executable,'-u',SOURCE/'run_csa_per_caption_test.py','prepare','--config',CONFIG],1800)
    cfg['worker_seconds']=min(1800,PARENT_DEADLINE-time.monotonic()-1800);write_config()
    if cfg['worker_seconds']<300:raise TimeoutError('Insufficient budget before TEST inference; no automatic extension')
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    for path,digest in {'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'experiments/csa_paper/scst_pilot_model.py': '0b5e2d384ffc7195c82888f3f7904eac145ba7291ebed27dc8342ed8e1ac9576'}.items():
        target=SOURCE/path if path.startswith('captioning/') else SOURCE/Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/cd3c8eb372f39df4454a2da27735e00e28f7f394/'+path,digest,target)
    pilot=SOURCE/'run_csa_per_caption_scst_pilot.py'
    pilot.write_text('"""User-run 500-step SCST + XE for CSA patch captioner."""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\nfrom full_data import FeatureReader, load_stage, prepare\nfrom real_bank_data import read, save, sha\nfrom semantic_data import SemanticContext, checked_root, record_arrays\n\nUPDATES = 500\nBATCH = 8\n\n\ndef prepare_train(cfg):\n    local = dict(cfg, stage=cfg[\'train_stage\'])\n    prepare(local)  # Metadata only.\n    stage, out = Path(local[\'stage\']), Path(cfg[\'output\'])\n    data, _, _ = load_stage(stage)\n    refs = {int(r[\'cocoid\']): [s[\'raw\'] for s in r[\'sentences\'][:5]]\n            for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\'] in (\'train\', \'restval\')}\n    if len(refs) != 113287 or set(refs) != set(map(int, data[\'ids\'][:113287])):\n        raise ValueError(\'Wrong TRAIN reward cohort\')\n    selected = np.random.default_rng(42).permutation(113287)[:UPDATES*BATCH]\n    ids = data[\'ids\'][selected]\n    first = np.searchsorted(data[\'caption_image\'], selected)\n    sequences = data[\'sequences\'][first]\n    inv = read(Path(__file__).with_name(\'FULL_PROMPT_FRESH_V175_RESULT.json\'))[\'transport_inventory\'][\'cache_files\']\n    files = {n: inv[n] for n in (\'cache_receipt.json\', \'cohort.json\', \'prompt_links.jsonl\',\n                                \'prompt_texts.json\', \'prompt_embeddings.npy\')}\n    root = checked_root(cfg[\'prompt_test_root\'], \'cache_receipt.json\', files)\n    cohort = read(root/\'cohort.json\')\n    if cohort[\'test\'] or set(cohort[\'train\']) != set(refs):\n        raise ValueError(\'Wrong TRAIN prompt cohort\')\n    texts = read(root/\'prompt_texts.json\')\n    vectors = np.load(root/\'prompt_embeddings.npy\', allow_pickle=False)\n    records, wanted = {}, set(map(int, ids))\n    with (root/\'prompt_links.jsonl\').open(encoding=\'utf8\') as stream:\n        for line in stream:\n            row = json.loads(line)\n            cid = row[\'coco_id\']\n            if cid in wanted:\n                if row[\'group\'] != \'train\' or cid in records:\n                    raise ValueError(\'Wrong TRAIN prompt row\')\n                records[cid] = record_arrays(row, texts)\n    if set(records) != wanted:\n        raise ValueError(\'Missing TRAIN prompts\')\n    for key in records[int(ids[0])]:\n        np.save(stage/f\'sem_{key}.npy\', np.stack([records[int(i)][key] for i in ids]))\n    np.save(stage/\'sem_text_vectors.npy\', vectors)\n    reader = FeatureReader(stage)\n    patches = np.lib.format.open_memmap(stage/\'pilot_patches.npy\', mode=\'w+\', dtype=np.float16,\n                                        shape=(len(ids), 196, 768))\n    try:\n        locations = data[\'locations\'][selected]\n        order = np.lexsort((locations[:, 1], locations[:, 0]))\n        for j, idx in enumerate(order):\n            patches[idx] = reader.get(selected[idx])\n            if (j+1) % 500 == 0:\n                print(\'Pilot TRAIN patches\', j+1, \'/\', len(ids), flush=True)\n        patches.flush()\n    finally:\n        reader.close()\n    np.save(stage/\'pilot_ids.npy\', ids)\n    np.save(stage/\'pilot_sequences.npy\', sequences)\n    save(stage/\'training_references.json\', refs)\n    save(out/\'pilot_inputs.json\', dict(updates=UPDATES, image_events=len(ids),\n         reward_images=len(refs), selected_ids=ids.tolist(), test_used=False))\n\n\ndef train(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch import nn\n    from torch.nn import functional as F\n    from datetime import timedelta\n    from captioning.scst import TrainingCider\n    from csa_patch_model import CSAPatchCaptioner\n    from csa_per_caption_test_data import checkpoint, validate_checkpoint_state, configure_valid\n    from scst_pilot_model import action_logits, policy_loss, captions, selected_logps\n\n    rank = int(os.environ[\'LOCAL_RANK\'])\n    if int(os.environ[\'WORLD_SIZE\']) != 2 or torch.cuda.device_count() != 2:\n        raise ValueError(\'Select T4 x2\')\n    torch.set_num_threads(1)\n    torch.cuda.set_device(rank)\n    device = torch.device(\'cuda\', rank)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.manual_seed(4242+rank)\n    dist.init_process_group(\'nccl\', timeout=timedelta(seconds=900))\n    out, stage = Path(cfg[\'output\']), Path(cfg[\'train_stage\'])\n    start = time.monotonic()\n\n    class Policy(nn.Module):\n        def __init__(self, model):\n            super().__init__()\n            self.model = model\n\n        def forward(self, x, p, sample, teacher):\n            memory = self.model.encode(x, p)\n            return self.model.decode(memory, sample), self.model.decode(memory, teacher)\n\n    @torch.no_grad()\n    def rollout(model, memory, sample):\n        seq = torch.ones((len(memory), 1), dtype=torch.long, device=device)\n        active = torch.ones(len(memory), dtype=torch.bool, device=device)\n        masks, selected = [], []\n        for _ in range(20):\n            logits = model.decode(memory, seq)[:, -1]\n            if not torch.isfinite(logits).all():\n                raise ValueError(\'Nonfinite rollout logits\')\n            logp = F.log_softmax(action_logits(logits), -1)\n            token = torch.multinomial(logp.exp(), 1).squeeze(1) if sample else logp.argmax(-1)\n            selected.append(logp.gather(1, token[:, None]).squeeze(1).masked_fill(~active, 0))\n            masks.append(active)\n            seq = torch.cat((seq, token.masked_fill(~active, 0)[:, None]), 1)\n            active = active & token.ne(2)\n            if not active.any():\n                break\n        return seq, torch.stack(masks, 1), torch.stack(selected, 1)\n\n    try:\n        configure_valid(cfg)\n        src = checkpoint(cfg[\'checkpoint_root\'])\n        ck = torch.load(src, map_location=\'cpu\', weights_only=False)\n        validate_checkpoint_state(ck)\n        words = read(stage/\'vocabulary.json\')\n        model = CSAPatchCaptioner(len(words), ck[\'model\'][\'core.nouns\'], ck[\'model\'][\'core.verbs\'])\n        model.load_state_dict(ck[\'model\'], strict=True)\n        del ck\n        model.to(device).eval()\n        wrapped = nn.parallel.DistributedDataParallel(Policy(model).eval(), device_ids=[rank], broadcast_buffers=False)\n        optimizer = torch.optim.Adam(wrapped.parameters(), lr=5e-6, betas=(.9,.99), eps=1e-8)\n        refs = {int(i): v for i, v in read(stage/\'training_references.json\').items()}\n        cider = TrainingCider(refs, reference_cache_size=2048)\n        del refs\n        context = SemanticContext(stage)\n        patches = np.load(stage/\'pilot_patches.npy\', mmap_mode=\'r\', allow_pickle=False)\n        sequences = np.load(stage/\'pilot_sequences.npy\', allow_pickle=False)\n        ids = np.load(stage/\'pilot_ids.npy\', allow_pickle=False)\n\n        def checkpoint_step(step):\n            if rank == 0:\n                path = out/\'pilot_checkpoint.bin\'\n                tmp = path.with_suffix(\'.tmp\')\n                torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),\n                                updates=step, source_sha256=sha(src)), tmp)\n                os.replace(tmp, path)\n                save(out/\'pilot_checkpoint_receipt.json\', dict(sha256=sha(path), updates=step,\n                     source_sha256=sha(src)))\n            dist.barrier(device_ids=[rank])\n\n        for step in range(UPDATES):\n            stop = torch.tensor(int(time.monotonic()-start > cfg[\'train_seconds\']-120), device=device)\n            dist.all_reduce(stop, op=dist.ReduceOp.MAX)\n            if int(stop):\n                checkpoint_step(step)\n                raise TimeoutError(\'Pilot cap reached; partial checkpoint preserved\')\n            rows = np.arange(step*BATCH+rank*4, step*BATCH+rank*4+4)\n            x = torch.from_numpy(np.asarray(patches[rows], np.float32)).to(device)\n            p = context.images(rows, device)\n            ce = np.asarray(sequences[rows], np.int64)\n            ce = torch.from_numpy(ce[:, :int((ce != 0).sum(1).max())]).to(device)\n            with torch.no_grad():\n                memory = model.encode(x, p)\n                sample, mask, replay = rollout(model, memory, True)\n                greedy, _, _ = rollout(model, memory, False)\n            sampled = cider.score(ids[rows].tolist(), captions(sample, words))\n            baseline = cider.score(ids[rows].tolist(), captions(greedy, words))\n            advantage = torch.from_numpy(sampled-baseline).to(device)\n            optimizer.zero_grad(set_to_none=True)\n            logits, teacher = wrapped(x, p, sample[:, :-1], ce[:, :-1])\n            if step == 0:\n                rescored = selected_logps(logits.detach(), sample[:, 1:], mask)\n                torch.testing.assert_close(rescored, replay, rtol=1e-4, atol=3e-4)\n            rl = policy_loss(logits, sample[:, 1:], mask, advantage)\n            nll = F.cross_entropy(teacher.flatten(0,1), ce[:,1:].flatten(), ignore_index=0, reduction=\'sum\')\n            tokens = ce[:,1:].ne(0).sum()\n            dist.all_reduce(tokens)\n            loss = nll*(2/int(tokens)) + .05*rl\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite SCST loss\')\n            loss.backward()\n            norm = nn.utils.clip_grad_norm_(wrapped.parameters(), 1., error_if_nonfinite=True)\n            optimizer.step()\n            if (step+1) % 25 == 0:\n                stats = torch.tensor([float(nll.detach()), float(rl.detach()), float(sampled.mean()),\n                                      float(baseline.mean())], dtype=torch.float64, device=device)\n                dist.all_reduce(stats)\n                if rank == 0:\n                    row = dict(updates=step+1, XE=float(stats[0])/int(tokens), SCST=float(stats[1])/2,\n                               sample_reward=float(stats[2])/2, greedy_reward=float(stats[3])/2,\n                               grad_norm=float(norm), seconds=time.monotonic()-start)\n                    print(\'SCST pilot\', row, flush=True)\n                    with (out/\'training.jsonl\').open(\'a\', encoding=\'utf8\') as log:\n                        log.write(json.dumps(row, allow_nan=False)+\'\\n\')\n            if (step+1) % 100 == 0:\n                checkpoint_step(step+1)\n    finally:\n        dist.destroy_process_group()\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'mode\', choices=(\'prepare_train\',\'train\'))\n    parser.add_argument(\'config\')\n    args = parser.parse_args()\n    globals()[args.mode](read(args.config))\n',encoding='utf8')

    run([sys.executable,'-u',pilot,'prepare_train',CONFIG],1800)
    cfg['train_seconds']=min(3600,PARENT_DEADLINE-time.monotonic()-1800)
    if cfg['train_seconds']<300:raise TimeoutError('Insufficient pilot time')
    write_config()
    run([*launch,pilot,'train',CONFIG],cfg['train_seconds']+30)
    cfg['evaluate_pilot']=True
    cfg['worker_seconds']=min(1800,PARENT_DEADLINE-time.monotonic()-600)
    if cfg['worker_seconds']<300:raise TimeoutError('Insufficient VALID evaluation time')
    write_config()
    evaluator=SOURCE/'run_csa_per_caption_test.py'
    run([*launch,evaluator,'worker','--config',CONFIG],cfg['worker_seconds']+30)
    for mode,cap in (('tokenize',180),('basic',240),('chair',600)):
        run([sys.executable,'-u',evaluator,mode,'--config',CONFIG],cap)
    metrics={}
    for mode in ('basic','chair'):
        metrics.update(json.loads((OUTPUT/(mode+'.json')).read_text())['metrics'])
    receipt=json.loads((OUTPUT/'pilot_checkpoint_receipt.json').read_text())
    if receipt['updates']!=500:raise ValueError('Incomplete pilot')
    reference=dict(CIDEr=1.202954787733369,CHAIRs=.04,CHAIRi=.027915291529152914,Recall=.4269839276745354)
    (OUTPUT/'result.json').write_text(json.dumps(dict(status='CSA_SCST_XE_PILOT500_VALID_COMPLETE',
        updates=500,images=5000,split='val',test_used=False,reranking=False,metrics=metrics,reference=reference,
        delta={k:metrics[k]-v for k,v in reference.items()},checkpoint=receipt),indent=2))
    complete=True
except BaseException:
    (OUTPUT/'failure.txt').write_text(redact(traceback.format_exc()));raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=True,test_used=False,test_tuning=False,source_version='csa_structure_per_caption_full_xe10_resume124177')))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP OUTPUT:',OUTPUT,flush=True)
    print('SCST+XE 500 and VALID complete.' if complete else 'INCOMPLETE; preserve captions/logs; no automatic re-decode or retraining.',flush=True)
